# 🎨 Flux2 Klein Image Generator — Colab Notebook

Run the **Flux2 Klein 9B** image generation app on a free **Google Colab T4** GPU (15 GB VRAM).

### Configuration
- **Transformer**: FP8 quantized (~8 GB VRAM)
- **Text Encoder**: GGUF Q8_0 Qwen3-VL 8B (~8 GB, group-offloaded)
- **Low VRAM Mode**: Smart group offloading with CUDA streams

> Total peak VRAM usage: ~8-9 GB (fits on T4!)
> 
> All model downloads run **in parallel** for maximum speed.

## 1️⃣ Check GPU

In [1]:
!nvidia-smi

## 2️⃣ Clone the Repository

In [2]:
!git clone https://github.com/vhp90/flxapp.git /content/flux2
%cd /content/flux2

## 3️⃣ Install Dependencies

In [3]:
# Install aria2 for fast multi-connection CivitAI downloads (16 parallel connections)
!apt-get install -y -qq aria2

# Install all project dependencies
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
%pip install -q -r requirements.txt
# gguf is needed for loading GGUF text encoder weights
%pip install -q gguf

## 4️⃣ Configure Environment

Set all environment variables for **low VRAM T4** mode.

⚠️ **Add your tokens below** (CivitAI token is required for downloading the transformer model).

In [ ]:
import os

# ============================================================
# 🔑 TOKENS — Fill these in!
# ============================================================
os.environ["HF_TOKEN"] = ""
os.environ["CIVITAI_TOKEN"] = ""  # Required: for downloading the transformer

# ============================================================
# Transformer: FP8 quantized Flux2 Klein 9B (~8 GB)
# ============================================================
os.environ["CIVITAI_MODEL_VERSION_ID"] = "2759272"
os.environ["TRANSFORMER_PATH"] = "/content/flux2/models/transformer_fp8.safetensors"
os.environ["TRANSFORMER_DTYPE"] = "fp8"

# ============================================================
# Text Encoder: GGUF Q8_0 Qwen3-8B abliterated (group-offloaded)
# ============================================================
os.environ["TEXT_ENCODER_ID"] = "mradermacher/Huihui-Qwen3-8B-abliterated-v2-GGUF"
os.environ["TEXT_ENCODER_GGUF_FILE"] = "Huihui-Qwen3-8B-abliterated-v2.Q8_0.gguf"
os.environ["TEXT_ENCODER_TOKENIZER_ID"] = "huihui-ai/Huihui-Qwen3-8B-abliterated-v2"

# ============================================================
# VAE & Scheduler from official Flux2 repo
# ============================================================
os.environ["FLUX2_REPO_ID"] = "black-forest-labs/FLUX.2-klein-9B"

# ============================================================
# Low VRAM + Downloads + Pipeline
# ============================================================
os.environ["LOW_VRAM_MODE"] = "true"
os.environ["ALLOW_MODEL_DOWNLOADS"] = "true"
os.environ["AUTO_INITIALIZE_PIPELINE"] = "true"
os.environ["ENABLE_MOCK_GENERATION"] = "false"

# ============================================================
# Generation defaults (conservative for T4)
# ============================================================
os.environ["DEFAULT_WIDTH"] = "768"
os.environ["DEFAULT_HEIGHT"] = "768"
os.environ["DEFAULT_STEPS"] = "4"
os.environ["DEFAULT_GUIDANCE_SCALE"] = "1.0"

# ============================================================
# Directories
# ============================================================
os.environ["OUTPUT_DIR"] = "/content/flux2/outputs"
os.environ["LORA_DIR"] = "/content/flux2/loras"

# Server
os.environ["HOST"] = "0.0.0.0"
os.environ["PORT"] = "7860"

# Enable fast HF downloads
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("✅ Environment configured for T4 Low VRAM mode")

## 5️⃣ Create Directories

In [5]:
!mkdir -p /content/flux2/models /content/flux2/outputs /content/flux2/loras
print("✅ Directories created")

## 6️⃣ Launch the Server

This will:
1. Download ALL models **in parallel** (transformer + text encoder + VAE simultaneously)
2. CivitAI uses aria2c (16 connections), HuggingFace uses hf_transfer (Rust parallel)
3. Load the pipeline with **smart group offloading** + CUDA streams
4. Start the web server on port 7860

⏱️ First run takes **5-10 minutes** for parallel downloads. Subsequent runs are instant.

In [6]:
# Start the server in the background
import subprocess, time, threading, os

# Kill any existing server on port 7860 to prevent 'address already in use' errors
!fuser -k 7860/tcp || true
time.sleep(1)


def run_server():
    proc = subprocess.Popen(
        ["python", "-m", "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "7860"],
        cwd="/content/flux2",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in iter(proc.stdout.readline, ""):
        print(line, end="")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to be ready
import urllib.request
for i in range(600):  # Wait up to 10 minutes for model downloads + loading
    time.sleep(1)
    try:
        resp = urllib.request.urlopen("http://localhost:7860/api/status")
        import json
        status = json.loads(resp.read())
        if status.get("ready"):
            print("\n" + "="*60)
            print("✅ SERVER IS READY!")
            print("="*60)
            break
        elif status.get("state") == "error":
            print(f"\n❌ Pipeline error: {status.get('error')}")
            break
    except:
        if i % 30 == 0 and i > 0:
            print(f"⏳ Still waiting... ({i}s elapsed)")
else:
    print("⏰ Timeout waiting for server. Check logs above for errors.")

## 7️⃣ Create Public URL with localtunnel

Use **localtunnel** (no signup needed) to expose the Colab server:

In [ ]:
# localtunnel (no signup)
!npm install -g localtunnel 2>/dev/null

import subprocess, re, time

lt_proc = subprocess.Popen(
    ["lt", "--port", "7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(3)
output = ""
while True:
    line = lt_proc.stdout.readline()
    if not line:
        break
    output += line
    if "url" in line.lower() or "https://" in line:
        print("\n" + "="*60)
        print("🌐 YOUR PUBLIC URL:")
        urls = re.findall(r'https://[\w.-]+\.loca\.lt', line)
        if urls:
            print(f"   {urls[0]}")
        else:
            print(f"   {line.strip()}")
        print("="*60)
        print("\nOpen this URL in your browser to use the Flux2 generator!")
        print("Note: You may need to click through a tunnel warning page first.")
        break

## 📸 Quick Test: Generate an Image via API

You can also generate images directly from this notebook:

In [ ]:
import requests, json
from IPython.display import Image, display

response = requests.post("http://localhost:7860/api/generate", json={
    "prompt": "a beautiful sunset over the ocean, photorealistic, 8k",
    "width": 768,
    "height": 768,
    "num_inference_steps": 4,
    "guidance_scale": 1.0,
    "seed": 42
})

result = response.json()
print(f"Seed: {result['seed']}")
for img in result["images"]:
    img_url = f"http://localhost:7860{img['url']}"
    img_data = requests.get(img_url).content
    display(Image(data=img_data))
    print(f"Saved: {img['url']}")

## 🧹 Cleanup (Optional)

Free up disk space by removing downloaded models:

In [ ]:
# Uncomment to clean up:
# !rm -rf /content/flux2/models/*.safetensors
# !rm -rf ~/.cache/huggingface/hub/
# print("🧹 Cleaned up model files")

---

### VRAM Budget Breakdown (T4 — 15 GB)

| Component | Size | Offload Strategy |
|-----------|------|------------------|
| FP8 Transformer | ~8 GB | leaf-level group offload + CUDA stream |
| GGUF Q8 Text Encoder | ~8 GB | block-level group offload (4 layers/group, ~2 GB peak) |
| VAE | ~0.3 GB | leaf-level group offload + CUDA stream |
| **Peak GPU usage** | **~8-9 GB** | ✅ Fits on T4! |

### Speed Improvements

| Feature | Before | After |
|---------|--------|-------|
| Downloads | Sequential (sum of all) | **Parallel** (max of single) |
| CivitAI | wget fallback | **aria2c 16 connections** |
| HuggingFace | Standard | **hf_transfer** (Rust parallel) |
| GPU Offload | Sequential (1 layer at a time) | **Group offload** + CUDA streams |